# 05 · Pipeline de entrenamiento con Prefect

**Propósito.** Describir y comprobar el flow `tubecast-training-pipeline` (`src/entrenamiento/flow.py`), que coordina el recorrido completo: datos → preparación → HPO de tres familias → entrenamiento y evaluación de los mejores modelos → registro en MLflow / Unity Catalog → alias `champion`.

| # | Task | Responsabilidad | Salida |
|---|---|---|---|
| 1 | `1_cargar_y_validar_datos` | Lee/descarga el parquet crudo; valida columnas, nulos, negativos y duplicados | DataFrame de videos |
| 2 | `2_preparar_serie_y_particiones` | Serie diaria, ventanas de 14 lags, split temporal 70/15/15 | Particiones |
| 3 | `3_hpo_y_modelo_final_por_familia` (×3) | Optuna en `hpo-<familia>` con `trial-NNN`; reentrena y evalúa `<familia>-final`; registra el artefacto | Métricas y `model_uri` |
| 4 | `4_comparar_y_seleccionar` | Tabla comparativa (`artifacts/model_comparison.csv`) y elección por menor `test_rmse` | Ganador |
| 5 | `5_registrar_en_registry_y_asignar_champion` | Registra las 3 versiones con etiquetas, asigna `@champion`, guarda respaldo local | Metadata del champion |

In [1]:
import os, sys, json
from pathlib import Path

# Los notebooks se ejecutan desde la raíz del repositorio para compartir rutas con src/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [2]:
from src.entrenamiento.reportes import configurar_mlflow
cfg = configurar_mlflow()
print("Tracking URI :", cfg["tracking_uri"])
print("Registry URI :", cfg["registry_uri"])
print("Experimento  :", cfg["experimento"])
print("Modelo       :", cfg["modelo"], "@" + cfg["alias"])

Tracking URI : sqlite:///mlflow.db
Registry URI : sqlite:///mlflow.db
Experimento  : /Shared/pcd-otono-2026-proyecto-tubecast
Modelo       : workspace.default.pcd_otono_2026_tubecast @champion


## 1. Ejecución

El flow no se lanza automáticamente desde este notebook (tarda varios minutos y escribe en el workspace remoto). Comandos, desde la raíz del repositorio:

```bash
uv run prefect server start                 # terminal 1: UI en http://127.0.0.1:4200
uv run python -m src.entrenamiento.flow 20  # terminal 2: 20 trials por familia
```

`PREFECT_API_URL=http://127.0.0.1:4200/api` (en `.env`) hace que la ejecución quede registrada en el servidor y pueda revisarse en la UI. Para ejecutarlo aquí, cambia `EJECUTAR_FLOW` a `True`.

In [3]:
import inspect
from src.entrenamiento import flow as modulo_flow
print(inspect.getsource(modulo_flow.ejecutar_pipeline_entrenamiento))

EJECUTAR_FLOW = False
if EJECUTAR_FLOW:
    resumen = modulo_flow.ejecutar_pipeline_entrenamiento(n_trials=20)

@flow(name="tubecast-training-pipeline", log_prints=True)
def ejecutar_pipeline_entrenamiento(n_trials: int = 20) -> dict:
    tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")
    registry_uri = os.getenv("MLFLOW_REGISTRY_URI", tracking_uri)
    experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "/Shared/pcd-otono-2026-proyecto-tubecast")

    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_registry_uri(registry_uri)
    mlflow.set_experiment(experiment_name)

    df_raw = task_cargar_datos()
    particiones = task_preparar_datos(df_raw)

    resultados = []
    for familia in FAMILIAS:
        res = task_hpo_familia(familia=familia, particiones=particiones, n_trials=n_trials)
        resultados.append(res)

    ganador = task_comparar_modelos(resultados)
    resumen = task_registrar_modelos(resultados, ganador)
    print("Pipeline finalizado con éxito. Resumen del modelo campeón:", resumen)
    return resumen



## 2. Comprobación en Prefect

In [4]:
from prefect.client.orchestration import get_client
from prefect.client.schemas.filters import FlowFilter, FlowFilterName, FlowRunFilter, FlowRunFilterId
from prefect.client.schemas.sorting import FlowRunSort

try:
    async with get_client() as client:
        flow_runs = await client.read_flow_runs(
            flow_filter=FlowFilter(name=FlowFilterName(any_=["tubecast-training-pipeline"])),
            sort=FlowRunSort.START_TIME_DESC, limit=5,
        )
        display(pd.DataFrame([{"flow_run": fr.name, "estado": fr.state_name, "inicio": fr.start_time,
                               "duración_s": round(fr.total_run_time.total_seconds(), 1)} for fr in flow_runs]))
        ultimo = next(fr for fr in flow_runs if fr.state_name == "Completed")
        task_runs = await client.read_task_runs(flow_run_filter=FlowRunFilter(id=FlowRunFilterId(any_=[ultimo.id])))
    tareas = pd.DataFrame([{"task_run": t.name, "estado": t.state_name, "inicio": t.start_time,
                            "duración_s": round(t.total_run_time.total_seconds(), 1)} for t in task_runs]).sort_values("inicio")
    display(Markdown(f"Tasks del último flow run exitoso **{ultimo.name}**:"))
    display(tareas)
    assert (tareas["estado"] == "Completed").all() and len(tareas) == 7
    print("Las 7 task runs (1 + 1 + 3 familias + 1 + 1) terminaron en Completed: OK")
except Exception as e:
    print(f"No se pudo consultar el servidor de Prefect ({type(e).__name__}: {e}).")
    print("Inicia `uv run prefect server start` y ejecuta el flow con PREFECT_API_URL configurada.")

,flow_run,estado,inicio,duración_s
0,humongous-capuchin,Completed,2026-10-27 01:23:09.361227+00:00,67.40
1,passionate-chowchow,Failed,2026-10-27 01:21:28.699299+00:00,43.30


Tasks del último flow run exitoso **humongous-capuchin**:

,task_run,estado,inicio,duración_s
6,1_cargar_y_validar_datos-286,Completed,2026-10-27 01:23:15.896505+00:00,0.20
5,2_preparar_serie_y_particiones-a67,Completed,2026-10-27 01:23:16.109385+00:00,0.10
4,3_hpo_y_modelo_final_por_familia-96e,Completed,2026-10-27 01:23:16.220935+00:00,9.40
3,3_hpo_y_modelo_final_por_familia-386,Completed,2026-10-27 01:23:25.631890+00:00,20.70
2,3_hpo_y_modelo_final_por_familia-537,Completed,2026-10-27 01:23:46.378955+00:00,29.20
1,4_comparar_y_seleccionar-cc9,Completed,2026-10-27 01:24:15.617434+00:00,0.10
0,5_registrar_en_registry_y_asignar_champion-357,Completed,2026-10-27 01:24:15.722273+00:00,1.00


Las 7 task runs (1 + 1 + 3 familias + 1 + 1) terminaron en Completed: OK


## 3. Salidas del flow

In [5]:
comparacion = pd.read_csv("artifacts/model_comparison.csv")
display(comparacion)
metadata = json.loads(Path("artifacts/champion_metadata.json").read_text(encoding="utf-8"))
metadata

,familia,final_run_id,n_trials,val_rmse,test_rmse,test_mae,test_mape,best_params
0,random-forest,39ee1e5abe07465ca8795074d0706605,20,"1,813,871.26","2,168,829.41","1,621,174.87",160.10,"{""n_estimators"": 250, ""max_depth"": 8, ""min_samples_split"": 2, ""min_samples_leaf"": 1}"
1,gradient-boosting,6b24e4c607524668802f91765b5174b6,20,"1,934,964.12","2,257,517.10","1,675,973.32",125.60,"{""learning_rate"": 0.01443734657496408, ""max_iter"": 100, ""max_depth"": 8, ""max_leaf_nodes"": 55, ""min_samples_leaf"": 7,..."
2,ridge,c8fb34793368468c811c914cc9019351,20,"2,010,293.04","2,318,929.81","1,763,913.74",143.32,"{""alpha"": 417.9950770779908, ""solver"": ""auto""}"


{'model_name': 'workspace.default.pcd_otono_2026_tubecast',
 'alias': 'champion',
 'version': '2',
 'familia_ganadora': 'random-forest',
 'final_run_id': '39ee1e5abe07465ca8795074d0706605',
 'tracking_uri': 'sqlite:///mlflow.db',
 'test_rmse': 2168829.4104615403,
 'test_mae': 1621174.8672469454,
 'test_mape': 160.10312414866098}

In [6]:
from mlflow.tracking import MlflowClient
alias = MlflowClient().get_model_version_by_alias(cfg["modelo"], cfg["alias"])
assert str(alias.version) == str(metadata["version"]), "La metadata local no corresponde al alias actual"
assert alias.run_id == metadata["final_run_id"] or alias.tags.get("final_run_id") == metadata["final_run_id"]
print(f"artifacts/ corresponde a {cfg['modelo']} v{alias.version} ({metadata['familia_ganadora']}): OK")

artifacts/ corresponde a workspace.default.pcd_otono_2026_tubecast v2 (random-forest): OK


**Interpretación.** Una sola ejecución del flow produce todo lo que consumen las etapas siguientes: runs de HPO y finales en el experimento `/Shared/pcd-otono-2026-proyecto-tubecast`, tres versiones en Unity Catalog, el alias `champion` y un respaldo local con su procedencia. Las tasks separan responsabilidades: si falla la validación de datos el flow se detiene antes de entrenar, y la selección del ganador queda registrada en el log de Prefect.

**Limitaciones.** No se usan deployments ni schedules (fuera del alcance de esta entrega); las tres familias se entrenan en serie. Si el Registry remoto no está disponible, el flow termina con el respaldo local y lo indica con un *warning*, no con un fallo.

Siguiente: [06-api](06-api.ipynb).